# Revisão humana, aprovação e rollback

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="07_registry_governanca"


{'mode': 'READ_EXISTING',
 'proposal_id': None,
 'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'model_name': 'hi.demo.notebook.abdb8f09',
 'model_version': '1',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': True,
 'confirmation': 'EXECUTE_DEMO',
 'request_key': 'notebooks-abdb8f09',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 30,
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'request',
 'review_action': 'APPROVE',
 'reason': 'Isolated notebook approval acceptance',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': None}

{'state': 'READY',
 'write_enabled': True,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Modelos/versões e pedidos de revisão. Registro é candidato; aprovação exige token separado de operador e versão explícita.

In [2]:
import mlflow
mlflow.set_tracking_uri(cfg.tracking);client=mlflow.MlflowClient()
versions=client.search_model_versions("name='"+params.model_name+"'",max_results=100)
display(pd.DataFrame([dict(model=v.name,version=v.version,run_id=v.run_id,tags=v.tags) for v in versions]))
display(pd.DataFrame(call(cfg,"GET","/v1/governance/requests")))


,model,version,run_id,tags
0,hi.demo.notebook.abdb8f09,1,eb44663c5112462ea5b95fd92c204151,"{'solution_id': 'horarios-inteligentes', 'envi..."


""


2. Inspecionar artefatos e proveniência da versão selecionada. Esperado: hashes iguais; nenhuma aprovação implícita.

In [3]:
if params.model_version:
    from src.operations.training import trace_for
    from src.workbench.verification import verify_model_storage
    trace=trace_for(client,client.get_model_version(params.model_name,params.model_version))
    evidence["candidate"]=trace;evidence["storage"]=verify_model_storage(trace)
    display(trace);display(pd.DataFrame(evidence["storage"]))


{'run_id': 'eb44663c5112462ea5b95fd92c204151',
 'experiment_id': '1',
 'model_name': 'hi.demo.notebook.abdb8f09',
 'model_version': '1',
 'model_uri': 'models:/hi.demo.notebook.abdb8f09/1',
 'artifact_uri': 'models:/m-3c632036203740d982f1b29aa117263d',
 'artifact_sha256': '4268666504b258b661b9c6cc009bdf8c21d06fde4317fb389a585e633be3ad0b',
 'metrics': {'duration_seconds': 0.526552208000794,
  'cpu_seconds': 0.21726249399999986,
  'peak_rss_kib': 276812.0,
  'mae': 3.758630658273197,
  'rmse': 4.9142272796917865,
  'baseline_mae': 7.466666666666665,
  'baseline_rmse': 8.821497143283464},
 'training_execution_id': '4e63488e-a440-4fed-babe-052716fac7cc'}

,bucket,key,bytes,sha256,model_version,status
0,pucpr-poc,mlflow/1/models/m-3c632036203740d982f1b29aa117...,1336,c1456acc4e87b4087686717f5612d0a412cdf9fca4078a...,1,PASS
1,pucpr-poc,mlflow/1/models/m-3c632036203740d982f1b29aa117...,8671,4268666504b258b661b9c6cc009bdf8c21d06fde4317fb...,1,PASS
2,pucpr-poc,mlflow/1/models/m-3c632036203740d982f1b29aa117...,80,3f22b46a5a2293ca36743e1757d14e9968aa14f9d57a28...,1,PASS


3. Ação opt-in: solicitar revisão ou decidir pedido identificado. APPROVE/REVOKE/ROLLBACK requerem justificativa; reviewer vem da credencial, não do texto digitado.

In [4]:
if params.governance_action!="read":
    params.authorize()
    if params.governance_action=="request":
        if not params.model_version or len(params.reason)<8: raise ValueError("Versão e justificativa obrigatórias")
        evidence["review"]=call(cfg,"POST","/v1/governance/requests",dict(model_name=params.model_name,model_version=params.model_version,
            action=params.review_action,reason=params.reason,requested_by=params.requester))
    else:
        if not params.approval_id: raise ValueError("approval_id obrigatório")
        evidence["review"]=call(cfg,"POST",f"/v1/governance/requests/{params.approval_id}/review",
            dict(approve=params.governance_action=="approve",confirm="REVIEW_EXPLICIT_VERSION"),operator=True)
    display(evidence["review"])
else:
    evidence["review"]="NOT_EXECUTED: leitura"


{'approval_id': '261e3132-103f-472e-9978-4b6b9ab27006',
 'model_name': 'hi.demo.notebook.abdb8f09',
 'model_version': '1',
 'action': 'APPROVE',
 'reason': 'Isolated notebook approval acceptance',
 'requested_by': 'scientist-demo',
 'reviewed_by': None,
 'correlation_id': '476039dd-2b13-413b-9d6b-7d12add1acd2',
 'status': 'REQUESTED',
 'prior_version': None,
 'artifact_sha256': None,
 'run_id': None,
 'created_at_utc': '2026-11-04T16:30:17.274278+00:00',
 'reviewed_at_utc': None}

Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [5]:
display(report(NOTEBOOK_ID,evidence,limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação"]))


'/reports/v2/07_registry_governanca.json'